# 組合せ範疇文法 (CCG) で文を導出する (NLTK)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/15_parsing/ccg_nltk.ipynb)

NLP クックブック 15-22 ／ 構文解析

単語に「どんな項を取って何になるか」という範疇を割り当て、少数の組合せ規則だけで文を導出する CCG を試したい。

In [1]:
from nltk.ccg import chart, lexicon

lex = lexicon.fromstring(r"""
    :- S, NP, N
    I => NP
    John => NP
    Mary => NP
    books => NP
    saw => (S\NP)/NP
    likes => (S\NP)/NP
    hates => (S\NP)/NP
    the => NP/N
    man => N
    telescope => N
    with => ((S\NP)\(S\NP))/NP
    with => (N\N)/NP
    and => var\.,var/.,var
""")

# 関数適用だけを使う: X/Y Y → X (前方適用 >)、Y X\Y → X (後方適用 <)
parser = chart.CCGChartParser(lex, chart.ApplicationRuleSet)
chart.printCCGDerivation(next(parser.parse("I saw the man".split())))

 I       saw       the    man
 NP  ((S\NP)/NP)  (NP/N)   N
                 ------------->
                      NP
    -------------------------->
              (S\NP)
------------------------------<
              S


In [2]:
# 構造的曖昧性: with の範疇が 2 つあるので、導出も 2 通りになる
parses = list(parser.parse("I saw the man with the telescope".split()))
print(len(parses), "通り")
for p in parses:
    chart.printCCGDerivation(p)

2 通り
 I       saw       the    man     with      the    telescope
 NP  ((S\NP)/NP)  (NP/N)   N   ((N\N)/NP)  (NP/N)      N
                                          ------------------->
                                                  NP
                              ------------------------------->
                                           (N\N)
                         ------------------------------------<
                                          N
                 -------------------------------------------->
                                      NP
    --------------------------------------------------------->
                             (S\NP)
-------------------------------------------------------------<
                              S
 I       saw       the    man          with           the    telescope
 NP  ((S\NP)/NP)  (NP/N)   N   (((S\NP)\(S\NP))/NP)  (NP/N)      N
                 ------------->
                      NP
    -------------------------->
              (S\

In [3]:
# 関数合成と型繰り上げを加えると、「John likes and Mary hates」のような構成素でないまとまりを等位接続できる
print("関数適用のみ:", len(list(parser.parse("John likes and Mary hates books".split()))), "通り")
full = chart.CCGChartParser(lex, chart.DefaultRuleSet)
parses = list(full.parse("John likes and Mary hates books".split()))
print("合成・型繰り上げあり:", len(parses), "通り")
chart.printCCGDerivation(parses[0])

関数適用のみ: 0 通り
合成・型繰り上げあり: 2 通り
 John     likes                and             Mary     hates     books
  NP   ((S\NP)/NP)  ((_var0\.,_var0)/.,_var0)   NP   ((S\NP)/NP)   NP
------>T
(S/(S\NP))
------------------->B
      (S/NP)
                                              ------>T
                                            (S/(S\NP))
                                              ------------------->B
                                                    (S/NP)
                   ---------------------------------------------->
                                 ((S/NP)\.,(S/NP))
-----------------------------------------------------------------<
                             (S/NP)
------------------------------------------------------------------------>
                                   S


- `(S\NP)/NP` は「右に NP を取り、次に左に NP を取ると S になる」= 他動詞。文法の情報のほとんどが辞書 (語彙項目) に入っていて、規則は言語によらない少数のものだけ、というのが CCG の特徴。
- 範疇と意味表現 (ラムダ式) を対応させると、導出と同時に論理式が組み上がる。意味解析の章の「文を一階述語論理の式に変換して真偽を確かめる (NLTK)」と同じ考え方。
- 規則を増やすと、意味が同じ導出が大量に生まれる (擬似的曖昧性)。実用的なパーザは正規形の制約でこれを抑える。
- 実用的な CCG パーザには、英語と日本語に対応した depccg がある。日本語 CCGbank や、CCG 導出から論理式を作って推論する ccg2lambda と組み合わせて使われる。